## Arquitetura do modelo

A estratégia implementada baseia-se em uma arquitetura robusta de gestão de risco e alocação dinâmica, ancorada em quatro pilares matemáticos que atuam em conjunto para modular a exposição do portfólio.

**Quatro pilares principais:**
1. HMM com emissões GMM para capturar caudas
2. Rede de conectividade para risco sistêmico
3. Volatilidade da Volatilidade (VoV) para instabilidade de regime
4. Entropia de Shannon para medir incerteza

---

## Pilar 1 — HMM com GMM

O núcleo preditivo do modelo é um Hidden Markov Model (HMM), responsável por classificar o regime de mercado em três estados latentes $S_t \in \{0, 1, 2\}$, correspondentes a cenários de baixa, média e alta volatilidade.

A sofisticação desta abordagem reside no uso de Gaussian Mixture Models (GMM) para as probabilidades de emissão. Em vez de assumir que retornos e volatilidade seguem uma distribuição normal simples em cada estado, o modelo define a probabilidade de observar o vetor de dados $x_t$, dado o estado $S_t = i$, como uma combinação linear de duas normais multivariadas:

$$
P(x_t \mid S_t = i) = \sum_{k=1}^{2} w_{i,k} \, \mathcal{N}(x_t \mid \mu_{i,k}, \Sigma_{i,k})
$$

onde $w_{i,k}$ são os pesos da mistura. Essa estrutura flexível é fundamental para acomodar a leptocurtose (caudas grossas) e a assimetria inerentes aos retornos financeiros. Ao sobrepor distribuições, o modelo consegue mapear outliers e choques de mercado sem distorcer o ajuste central da distribuição, garantindo que eventos de cauda não sejam subestimados.

---

## Pilar 2 — Rede de conectividade (risco sistêmico)

Para capturar o risco sistêmico, o algoritmo modela o mercado como um grafo, construindo dinamicamente uma rede de conectividade entre os ativos da B3. Utilizando uma janela móvel de 21 dias, calcula-se a matriz de correlação dos log-retornos. Essa matriz é então filtrada por um limiar de corte rígido de 0,7 para gerar a matriz de adjacência $A$, onde as arestas indicam conexões direcionais severas.

A densidade da rede é quantificada pela razão entre o número de arestas ativas e o limite combinatório máximo de conexões possíveis no sistema, dado pelo coeficiente binomial $\binom{n}{2}$. Quando essa densidade ultrapassa o percentil de 80% de seu histórico, o modelo infere alta sincronia entre os ativos (comportamento de "manada") e aciona um corte defensivo de 50% na exposição de capital.

Simultaneamente, a rotina extrai o maior autovalor ($\lambda_{\max}$) da matriz de correlação original via decomposição espectral. O crescimento acelerado de $\lambda_{\max}$ indica que o autovetor principal está absorvendo a maior parte da variância do sistema, o que traduz uma concentração brutal de risco e a iminência de uma quebra estrutural no mercado.

---

## Pilar 3 — Volatilidade da Volatilidade (VoV)

Complementando essa topologia de rede, a Volatilidade da Volatilidade (VoV) atua como uma métrica de segunda derivada da dinâmica de preços. Se a volatilidade anualizada local $\sigma_t$ é o desvio padrão dos retornos, a VoV mede o desvio padrão da própria $\sigma_t$ em uma janela contínua.

Este indicador funciona como um acelerômetro: picos na VoV sinalizam que o regime de prêmio de risco está sofrendo mutações rápidas, acusando instabilidade antes que o HMM tenha dados suficientes para formalizar uma transição de estado.

---

## Pilar 4 — Entropia de Shannon

Por fim, o modelo utiliza a teoria da informação para auditar sua própria incerteza estatística por meio da Entropia de Shannon. A cada instante, o HMM não apenas decreta um estado, mas também fornece um vetor de probabilidades $p_i$ de o mercado pertencer a cada um dos três regimes. A incerteza distributiva dessa predição é medida por:

$$
H = -\sum_{i=1}^{3} p_i \log_2(p_i)
$$

Sabendo que a entropia máxima para um sistema de três estados equiprováveis é $\log_2(3) \approx 1{,}58$, o algoritmo impõe um limite de atuação. Se $H > 1{,}2$, a distribuição de probabilidades está excessivamente plana (o modelo está "confuso" ou o mercado atravessa uma zona de transição ruidosa). Nesse cenário de baixa convicção matemática, um modulador de exposição intervém e reduz pela metade o tamanho de todas as novas posições, blindando o patrimônio contra a imprecisão preditiva temporária.